Generiamo il grafo in formato .json per la visualizzazione

In [1]:
import networkx as nx
import json
from utils import *
import igraph as ig

In [2]:
graph_path = DATA_DIR + '\Centrality\Centrality_immigration_23.graphml'
G = load_graph(graph_path)
print(f"Attributi nodi disponibili: {G.vs.attributes()}")
print(f"Attributi archi disponibili: {G.es.attributes()}")



# Mapping degli attributi GraphML
attribute_mapping = {
    'v_hierarchy': 'hierarchy',
    'v_group': 'group',
    'v_id': 'original_id',
    'v_degree': 'degree_centrality',
    'v_eigenvector': 'eigenvector',
    'v_closeness': 'closeness',
    'v_betweenness': 'betweenness',
    'v_coreness': 'coreness'
}

# Costruisci i nodi
nodes = []
for i, vertex in enumerate(G.vs):
    node_obj = {'id': str(vertex['id']) if 'id' in vertex.attributes() else str(i)}
    
    # Estrai gli attributi dal GraphML
    for graphml_key, json_key in attribute_mapping.items():
        if graphml_key in vertex.attributes():
            try:
                # Converti in float dove possibile
                value = float(vertex[graphml_key])
            except (ValueError, TypeError):
                # Altrimenti mantieni come stringa
                value = str(vertex[graphml_key])
            node_obj[json_key] = value
        else:
            # Se l'attributo non esiste, metti un valore di default
            if json_key in ['degree', 'eigenvector', 'closeness', 'betweenness', 'coreness']:
                node_obj[json_key] = 0
    
    # Aggiungi tutti gli altri attributi disponibili
    for attr in vertex.attributes():
        if attr not in attribute_mapping:
            try:
                node_obj[attr] = float(vertex[attr])
            except (ValueError, TypeError):
                node_obj[attr] = str(vertex[attr])
    
    nodes.append(node_obj)

# Costruisci gli spigoli
links = []
for edge_idx, edge in enumerate(G.es):
    source_id = str(G.vs[edge.source]['id']) if 'id' in G.vs[edge.source].attributes() else str(edge.source)
    target_id = str(G.vs[edge.target]['id']) if 'id' in G.vs[edge.target].attributes() else str(edge.target)
    
    link_obj = {
        'id': edge_idx,
        'source': source_id,
        'target': target_id,
        'weight': 1.0
    }
    
    # Se ci sono attributi di peso o altri dati
    for attr in edge.attributes():
        try:
            link_obj[attr] = float(edge[attr])
        except (ValueError, TypeError):
            link_obj[attr] = str(edge[attr])
    
    # Se c'è un attributo 'weight', usalo
    if 'weight' in edge.attributes():
        link_obj['weight'] = float(edge['weight'])
    
    links.append(link_obj)

# Crea l'output JSON
output = {
    'nodes': nodes,
    'links': links,
    'metadata': {
        'num_nodes': len(nodes),
        'num_edges': len(links),
        'groups': list(set(n.get('group', 'N/A') for n in nodes if 'group' in n)),
        'density': G.density(),
        'diameter': G.diameter() if G.is_connected() else -1
    }
}

# Salva in JSON
with open('grafo.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"\n Grafo salvato: grafo.json")
print(f" Nodi: {len(nodes)}, Spigoli: {len(links)}")

# Calcola range di coreness
coreness_values = [n.get('coreness', 0) for n in nodes if n.get('coreness') is not None]
if coreness_values:
    print(f" Coreness range: {min(coreness_values):.0f} - {max(coreness_values):.0f}")

groups = output['metadata']['groups']
if groups and groups != ['N/A']:
    print(f" Gruppi trovati: {groups}")

print(f"\n Esempio di nodo:")
print(json.dumps(nodes[0], indent=2))

print(f"\n Metadati grafo:")
print(f"  - Densità: {output['metadata']['density']:.4f}")
print(f"  - Diametro: {output['metadata']['diameter']}")


c:\Users\maria\Desktop\Analisi e visualizzazione reti complesse\AVRC_FinnishTwittersphereProject\Code\utils.py:28: RuntimeWarning: Could not add vertex ids, there is already an 'id' vertex attribute. Location: src/io/graphml.c:434
  return ig.Graph.Read_GraphML(filename)


Attributi nodi disponibili: ['hierarchy', 'group', 'id', 'degree', 'eigenvector', 'closeness', 'betweenness', 'coreness']
Attributi archi disponibili: []

 Grafo salvato: grafo.json
 Nodi: 9519, Spigoli: 36889
 Coreness range: 1 - 25
 Gruppi trovati: ['B', 'A']

 Esempio di nodo:
{
  "id": 33725.0,
  "eigenvector": 0.0592978971467666,
  "closeness": 0.26989168037203,
  "betweenness": 2.03080618623589e-05,
  "coreness": 7.0,
  "hierarchy": "B_PERIPHERY",
  "group": "B",
  "degree": 7.0
}

 Metadati grafo:
  - Densità: 0.0008
  - Diametro: 13
